# STAGE 1 - IMPORTS AND CONFIGURATION

In [1]:
import os
from openai import OpenAI

# Set your API key directly (avoid sharing this code publicly)
os.environ["OPENAI_API_KEY"] = "Your API Key"

# Initialize the client
client = OpenAI()

In [2]:
client = OpenAI()
print("Client initialized successfully!")

Client initialized successfully!


In [3]:
import sys
import types

# Bypass the broken Ragas import
dummy_vertex = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_vertex.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_vertex

# Now import ragas normally below this line
import ragas

In [5]:
import os
import time
import asyncio
import logging
import numpy as np
import pandas as pd
import nltk
import instructor

from dotenv import load_dotenv
from openai import AsyncOpenAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings
from ragas.llms import InstructorLLM
from ragas.metrics.collections import AnswerCorrectness
import os
import time
import asyncio
import logging
import numpy as np
import pandas as pd
import nltk
import instructor

from dotenv import load_dotenv
from openai import AsyncOpenAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings
from ragas.llms import InstructorLLM
from ragas.metrics.collections import AnswerCorrectness
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric
)

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

logging.getLogger("deepeval").setLevel(logging.CRITICAL)
logging.getLogger("transformers").setLevel(logging.ERROR)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

INPUT_CSV = "ai_teacher_20_questions.csv"
PDF_FILE = "HopeAI_Teaching.pdf"
OUTPUT_CSV = "RAG_LLM_FINAL_EVALUATION.csv"

MODEL = "gpt-4o-mini"
EVALUATION_MODEL = "gpt-4.1"

# Configure these according to the current API pricing.
INPUT_PRICE = 0.15 / 1_000_000
OUTPUT_PRICE = 0.60 / 1_000_000

df = pd.read_csv(INPUT_CSV)

required_columns = ["User Query", "Teacher Response"]
missing = set(required_columns) - set(df.columns)

if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

df = df.dropna(subset=required_columns).reset_index(drop=True)

df["User Query"] = df["User Query"].astype(str).str.strip()
df["Teacher Response"] = df["Teacher Response"].astype(str).str.strip()

print(f"Questions loaded: {len(df)}")



Questions loaded: 20


/tmp/ipykernel_15587/2503749121.py:51: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [4]:
!pip install instructor

  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: jiter
    Found existing installation: jiter 0.18.0
    Uninstalling jiter-0.18.0:
      Successfully uninstalled jiter-0.18.0
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [instructor]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain_community]


In [6]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.1/587.1 kB 6.0 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain_openai]


In [7]:
!pip install langchain-community langchain-google-vertexai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 67.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-vertexai]


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 70.0 MB/s  0:00:00


In [8]:
!pip install bert_score

In [9]:
!pip install rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=cbc2c4e17bca8f614c76db9989a45345d5f826a5db2095ed18903d775a87b177
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [10]:
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 42.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ragas]


In [11]:
!pip install --upgrade ragas langchain-community

In [12]:
!pip install "langchain-community<0.3.0" langchain-google-vertexai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 73.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-google-vertexai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-google-vertexai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of google-cloud-aiplatform to determine which version is compatible with other requirements. This could take a while.
INFO: pip is

In [ ]:
!pip install --upgrade ragas langchain langchain-community langchain-google-vertexai

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_google_vertexai-3.2.5-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-1.8.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
INFO: pip is looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
  Using cached google_genai-1.75.0-py3-none-any.whl.metadata (52 kB)
Using cached langchain_core-1.8.0-py3-none-any.whl (587 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 22.8 MB/s  0:00:00
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_text_splitters-1.1.3-py3-none-any.whl (38 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 40.8 MB/s  0:00:00
Using cached langchain_google_vertexai-3.2.5-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793

In [1]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.4 MB/s  0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Successfully un

In [5]:
!pip install --upgrade langchain-core ragas langchain

  Using cached langchain_core-1.8.0-py3-none-any.whl.metadata (4.7 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain_core-1.8.0-py3-none-any.whl (587 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.8/940.8 kB 16.6 MB/s  0:00:00
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_text_splitters-1.1.3-py3-none-any.whl (38 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 56.3 MB/s  0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 14.2
    Uninstalling websockets-14.2:
      Successfully uninstalled websockets-14.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26

# STAGE 2 - RAG PIPELINE

In [11]:
# LOAD PDF
documents = PyPDFLoader(PDF_FILE).load()

# CHUNK DOCUMENTS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

# CREATE FAISS VECTOR STORE
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# LLM
gpt_model = ChatOpenAI(model=MODEL, temperature=0, timeout=120, max_retries=3)

# RAG PROMPT

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the retrieved context.

If the answer is not available in the context,
say that the information could not be found in the documents.

Context:
{context}

Question:
{question}

Answer:
""")

# RETRIEVE CONTEXT

def get_context(query):
    retrieved_docs = retriever.invoke(query)
    retrieval_context = [
        doc.page_content for doc in retrieved_docs
    ]
    context = "\n\n".join(retrieval_context)
    return context, retrieval_context

print("PDF pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("FAISS vector store and top-3 retriever ready.")

PDF pages loaded: 6
Chunks created: 18
FAISS vector store and top-3 retriever ready.


In [8]:
!pip install pypdf

In [10]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 MB 22.5 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 23.0 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 43.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 48.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [faiss-gpu]


# STAGE 3 - DEFINE EVALUATION METRICS

In [12]:
# SIMILARITY METRICS

bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# ANSWER QUALITY METRICS
correctness_metric = GEval(
    name="Answer Correctness",
    model=EVALUATION_MODEL,
    criteria="""
    Compare the generated answer with the reference answer.
    Evaluate factual accuracy, important information,
    and contradictions. Ignore minor wording differences.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

helpfulness_metric = GEval(
    name="Helpfulness",
    model=EVALUATION_MODEL,
    criteria="""
    Evaluate whether the answer directly addresses the question,
    is easy for a beginner to understand, and gives a clear,
    useful explanation without unnecessary information.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    async_mode=False
)
answer_relevancy_metric = AnswerRelevancyMetric(model=EVALUATION_MODEL, include_reason=False, async_mode=False)

# RAGAS ANSWER CORRECTNESS
ragas_client = AsyncOpenAI(api_key=api_key)
instructor_client = instructor.from_openai(ragas_client)
ragas_llm = InstructorLLM(
    client=instructor_client,
    model="gpt-5-nano",
    provider="openai",
    max_tokens=8192,
    temperature=1.0
)
ragas_embedding_client = AsyncOpenAI(api_key=api_key)
ragas_embedding_model = RagasEmbeddings(client=ragas_embedding_client, model="text-embedding-3-small")
answer_correctness_metric = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_embedding_model)

# RAG GEVAL
rag_geval_metric = GEval(
    name="RAG Correctness",
    model="gpt-5-nano",
    criteria="""
    Evaluate whether the generated answer is factually correct
    and consistent with the reference answer.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

# RETRIEVAL / RAG METRICS
context_precision_metric = ContextualPrecisionMetric(model=EVALUATION_MODEL)
context_recall_metric = ContextualRecallMetric(model=EVALUATION_MODEL)
faithfulness_metric = FaithfulnessMetric(model=EVALUATION_MODEL)
rag_answer_relevancy_metric = AnswerRelevancyMetric(model=EVALUATION_MODEL, include_reason=False, async_mode=False)
print("All evaluation metrics initialized.")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

All evaluation metrics initialized.


# STAGE 4 - INPUT QUERIES, GENERATE ANSWERS AND CALCULATE OPERATIONAL METRICS

In [13]:
from tqdm.auto import tqdm

results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="RAG generation"
):
    query = row["User Query"]
    reference = row["Teacher Response"]

    try:
        total_start = time.perf_counter()

        # RETRIEVAL
        start = time.perf_counter()
        context, retrieval_context = get_context(query)
        retrieval_latency = time.perf_counter() - start

        # GENERATION
        prompt_value = rag_prompt.invoke({
            "context": context,
            "question": query
        })

        start = time.perf_counter()
        response = gpt_model.invoke(prompt_value)
        generation_latency = time.perf_counter() - start

        total_latency = time.perf_counter() - total_start

        generated = response.content.strip()

        # TOKEN USAGE
        metadata = getattr(response, "response_metadata", {})
        usage = metadata.get("token_usage", {})

        input_tokens = usage.get("prompt_tokens", 0)
        output_tokens = usage.get("completion_tokens", 0)
        total_tokens = usage.get(
            "total_tokens",
            input_tokens + output_tokens
        )

        # COST
        input_cost = input_tokens * INPUT_PRICE
        output_cost = output_tokens * OUTPUT_PRICE
        total_cost = input_cost + output_cost

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": generated,
            "RetrievedContext": retrieval_context,

            "RetrievalLatency": retrieval_latency,
            "GenerationLatency": generation_latency,
            "Latency": total_latency,

            "InputTokens": input_tokens,
            "OutputTokens": output_tokens,
            "TotalTokens": total_tokens,

            "InputCost": input_cost,
            "OutputCost": output_cost,
            "TotalCost": total_cost
        })

    except Exception as e:
        print(f"\nError: {query}\n{e}")

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": "",
            "RetrievedContext": [],

            "RetrievalLatency": np.nan,
            "GenerationLatency": np.nan,
            "Latency": np.nan,

            "InputTokens": np.nan,
            "OutputTokens": np.nan,
            "TotalTokens": np.nan,

            "InputCost": np.nan,
            "OutputCost": np.nan,
            "TotalCost": np.nan
        })


results_df = pd.DataFrame(results)

display(results_df)

RAG generation:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,Teacher Response,GeneratedAnswer,RetrievedContext,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"AI, or artificial intelligence, is the ability...",[when actions are ambiguous or poorly controll...,0.962801,2.734779,3.699158,534,72,606,0.000080,0.000043,0.000123
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,[knowledge for a small educational chatbot or ...,0.314274,1.944034,2.259051,533,87,620,0.000080,0.000052,0.000132
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,[The relationship among the three concepts is ...,0.559600,2.278406,2.838481,394,58,452,0.000059,0.000035,0.000094
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,[The relationship among the three concepts is ...,0.329637,2.353189,2.683331,447,47,494,0.000067,0.000028,0.000095
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,"The difference among AI, ML, and Deep Learning...",[The relationship among the three concepts is ...,0.242008,2.096741,2.339420,455,64,519,0.000068,0.000038,0.000107
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,0.291848,2.406575,2.698912,489,72,561,0.000073,0.000043,0.000117
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the mathematical s...,"[recommendation, or decision. The model does n...",0.221413,1.311056,1.532934,461,67,528,0.000069,0.000040,0.000109
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,"[recommendation, or decision. The model does n...",0.204123,1.984683,2.189297,406,64,470,0.000061,0.000038,0.000099
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,"[recommendation, or decision. The model does n...",0.559724,1.504315,2.064526,408,70,478,0.000061,0.000042,0.000103
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,"[model searches for hidden structures, similar...",0.292116,2.147465,2.440185,531,65,596,0.000080,0.000039,0.000119


# STAGE 5 - LLM EVALUATION - ANSWER QUALITY

In [14]:
quality_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Answer quality"
):
    query = row["User Query"]
    generated = row["GeneratedAnswer"]
    reference = row["Teacher Response"]

    if not generated:
        quality_scores.append([np.nan] * 3)
        continue

    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference
    )

    scores = []

    for metric in [
        correctness_metric,
        answer_relevancy_metric,
        helpfulness_metric
    ]:
        try:
            metric.measure(test_case)
            scores.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            scores.append(np.nan)

    quality_scores.append(scores)


results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
] = pd.DataFrame(
    quality_scores,
    index=results_df.index
)

results_df["AnswerQuality"] = results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
].mean(axis=1)

display(results_df[
    [
        "User Query",
        "AnswerCorrectness",
        "AnswerRelevancy",
        "Helpfulness",
        "AnswerQuality"
    ]
])

Answer quality:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

Output()

... (52 outputs left out)

Output()

Output()

Output()

,User Query,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,0.900000,1.0,1.000000,0.966667
1,How does AI differ from traditional programming?,0.919193,1.0,1.000000,0.973064
2,What is Machine Learning?,0.941490,1.0,1.000000,0.980497
3,What is Deep Learning?,0.901364,1.0,0.930075,0.943813
4,"What is the difference between AI, ML, and Dee...",0.995257,1.0,0.897482,0.964247
5,What is training data?,0.806560,1.0,0.997482,0.934681
6,What is a Machine Learning model?,0.903210,1.0,1.000000,0.967737
7,What is supervised learning?,0.998935,1.0,1.000000,0.999645
8,What is unsupervised learning?,0.974317,1.0,0.969264,0.981194
9,What is reinforcement learning?,0.955447,1.0,1.000000,0.985149


# STAGE 6 - EVALUATE SIMILARITY METRICS

In [15]:
similarity_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Similarity metrics"
):
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]

    if not generated:
        similarity_scores.append([np.nan] * 5)
        continue

    try:
        _, _, f1 = bert_scorer.score(
            [generated],
            [reference]
        )
        bert = float(f1[0])
    except Exception:
        bert = np.nan

    try:
        rouge_scores = rouge.score(reference, generated)

        rouge1 = rouge_scores["rouge1"].fmeasure
        rouge2 = rouge_scores["rouge2"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure
    except Exception:
        rouge1 = rouge2 = rougeL = np.nan

    try:
        meteor = meteor_score(
            [reference.split()],
            generated.split()
        )
    except Exception:
        meteor = np.nan

    similarity_scores.append([
        bert, rouge1, rouge2, rougeL, meteor
    ])


similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]

results_df[similarity_columns] = pd.DataFrame(
    similarity_scores,
    index=results_df.index
)

display(results_df[["User Query"] + similarity_columns])

Similarity metrics:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.936465,0.541176,0.385542,0.494118,0.697060
1,How does AI differ from traditional programming?,0.896392,0.326923,0.196078,0.288462,0.466625
2,What is Machine Learning?,0.909317,0.444444,0.278481,0.444444,0.457683
3,What is Deep Learning?,0.941287,0.606061,0.468750,0.606061,0.739874
4,"What is the difference between AI, ML, and Dee...",0.913300,0.597701,0.352941,0.528736,0.558382
5,What is training data?,0.895691,0.391304,0.200000,0.347826,0.465113
6,What is a Machine Learning model?,0.900434,0.450000,0.153846,0.300000,0.442387
7,What is supervised learning?,0.913420,0.457831,0.172840,0.337349,0.484613
8,What is unsupervised learning?,0.928887,0.439024,0.175000,0.365854,0.503774
9,What is reinforcement learning?,0.927300,0.459459,0.222222,0.351351,0.618849


# STAGE 7 - RAG EVALUATION METRICS

In [16]:
async def calculate_ragas(query, reference, generated):
    try:
        score = await answer_correctness_metric.ascore(
            user_input=query,
            response=generated,
            reference=reference
        )
        return float(score.value)
    except Exception as e:
        print("RAGAS error:", e)
        return np.nan


rag_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="RAG evaluation"
):
    query = row["User Query"]
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]
    context = row["RetrievedContext"]

    if not generated:
        rag_scores.append([np.nan] * 6)
        continue

    # RAGAS
    ragas_score = await calculate_ragas(
        query, reference, generated
    )

    # DEEPEVAL RAG METRICS
    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference,
        retrieval_context=context
    )

    metric_values = []

    for metric in [
        context_precision_metric,
        context_recall_metric,
        faithfulness_metric,
        rag_answer_relevancy_metric,
        rag_geval_metric
    ]:
        try:
            metric.measure(test_case)
            metric_values.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            metric_values.append(np.nan)

    # Order: RAGAS, precision, recall, faithfulness,
    # answer relevancy, GEval
    rag_scores.append([
        ragas_score,
        *metric_values
    ])


rag_columns = [
    "Ragas AnswerCorrectness",
    "Context Precision",
    "Context Recall",
    "Faithfulness",
    "RAG Answer Relevancy",
    "RAG GEval"
]

results_df[rag_columns] = pd.DataFrame(
    rag_scores,
    index=results_df.index
)

results_df["RAG Evaluation Score"] = results_df[
    rag_columns
].mean(axis=1)

display(results_df[["User Query"] + rag_columns + [
    "RAG Evaluation Score"
]])

RAG evaluation:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

Output()

... (94 outputs left out)

Output()

Output()

Output()

,User Query,Ragas AnswerCorrectness,Context Precision,Context Recall,Faithfulness,RAG Answer Relevancy,RAG GEval,RAG Evaluation Score
0,What is AI?,0.792442,0.5,1.0,1.0,1.00,0.5,0.798740
1,How does AI differ from traditional programming?,0.778221,1.0,1.0,1.0,1.00,0.6,0.896370
2,What is Machine Learning?,0.779266,1.0,1.0,1.0,1.00,0.7,0.913211
3,What is Deep Learning?,0.863616,1.0,1.0,1.0,1.00,0.7,0.927269
4,"What is the difference between AI, ML, and Dee...",0.841817,1.0,1.0,1.0,1.00,1.0,0.973636
5,What is training data?,0.358796,1.0,1.0,1.0,1.00,0.6,0.826466
6,What is a Machine Learning model?,0.715267,1.0,1.0,1.0,1.00,0.7,0.902544
7,What is supervised learning?,0.840417,1.0,1.0,1.0,1.00,0.7,0.923403
8,What is unsupervised learning?,0.792037,1.0,1.0,1.0,1.00,0.7,0.915340
9,What is reinforcement learning?,0.588110,1.0,1.0,1.0,1.00,0.7,0.881352


# STAGE 8 - OPERATIONAL METRICS SUMMARY

In [17]:
operational_columns = [
    "RetrievalLatency",
    "GenerationLatency",
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

operational_summary = pd.DataFrame({
    "Metric": operational_columns,
    "Average": [
        results_df[c].mean() for c in operational_columns
    ],
    "Total": [
        results_df[c].sum() for c in operational_columns
    ]
})

display(operational_summary.round(6))

,Metric,Average,Total
0,RetrievalLatency,0.329380,6.587610
1,GenerationLatency,1.960055,39.201103
2,Latency,2.290017,45.800332
3,InputTokens,472.300000,9446.000000
4,OutputTokens,66.900000,1338.000000
5,TotalTokens,539.200000,10784.000000
6,InputCost,0.000071,0.001417
7,OutputCost,0.000040,0.000803
8,TotalCost,0.000111,0.002220


# STAGE 9 - FINAL EVALUATION TABLE SUMMARY

In [18]:
# ROUND ALL METRIC COLUMNS

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality",
    *similarity_columns,
    *rag_columns,
    "RAG Evaluation Score"
]

results_df[metric_columns] = results_df[metric_columns].round(3)


# FINAL QUESTION-WISE TABLE

final_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer",
    *metric_columns,
    *operational_columns
]

final_df = results_df[final_columns].copy()

display(final_df)


# AVERAGE EVALUATION SUMMARY
summary_df = pd.DataFrame({
    "Metric": metric_columns,
    "Average Score": [
        results_df[c].mean() for c in metric_columns
    ]
})

summary_df["Average Score"] = (
    summary_df["Average Score"].round(3)
)

display(summary_df)


# SAVE OUTPUTS

final_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "RAG_LLM_EVALUATION_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

operational_summary.to_csv(
    "RAG_LLM_OPERATIONAL_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation completed.")
print("Final results:", OUTPUT_CSV)
print("Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv")
print("Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv")

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality,BERT,ROUGE-1,ROUGE-2,...,RAG Evaluation Score,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"AI, or artificial intelligence, is the ability...",0.900,1.0,1.000,0.967,0.936,0.541,0.386,...,0.799,0.962801,2.734779,3.699158,534,72,606,0.000080,0.000043,0.000123
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,0.919,1.0,1.000,0.973,0.896,0.327,0.196,...,0.896,0.314274,1.944034,2.259051,533,87,620,0.000080,0.000052,0.000132
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,0.941,1.0,1.000,0.980,0.909,0.444,0.278,...,0.913,0.559600,2.278406,2.838481,394,58,452,0.000059,0.000035,0.000094
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,0.901,1.0,0.930,0.944,0.941,0.606,0.469,...,0.927,0.329637,2.353189,2.683331,447,47,494,0.000067,0.000028,0.000095
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,"The difference among AI, ML, and Deep Learning...",0.995,1.0,0.897,0.964,0.913,0.598,0.353,...,0.974,0.242008,2.096741,2.339420,455,64,519,0.000068,0.000038,0.000107
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,0.807,1.0,0.997,0.935,0.896,0.391,0.200,...,0.826,0.291848,2.406575,2.698912,489,72,561,0.000073,0.000043,0.000117
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the mathematical s...,0.903,1.0,1.000,0.968,0.900,0.450,0.154,...,0.903,0.221413,1.311056,1.532934,461,67,528,0.000069,0.000040,0.000109
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,0.999,1.0,1.000,1.000,0.913,0.458,0.173,...,0.923,0.204123,1.984683,2.189297,406,64,470,0.000061,0.000038,0.000099
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,0.974,1.0,0.969,0.981,0.929,0.439,0.175,...,0.915,0.559724,1.504315,2.064526,408,70,478,0.000061,0.000042,0.000103
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,0.955,1.0,1.000,0.985,0.927,0.459,0.222,...,0.881,0.292116,2.147465,2.440185,531,65,596,0.000080,0.000039,0.000119


,Metric,Average Score
0,AnswerCorrectness,0.913
1,AnswerRelevancy,0.995
2,Helpfulness,0.969
3,AnswerQuality,0.959
4,BERT,0.915
5,ROUGE-1,0.452
6,ROUGE-2,0.253
7,ROUGE-L,0.385
8,METEOR,0.504
9,Ragas AnswerCorrectness,0.784


Evaluation completed.
Final results: RAG_LLM_FINAL_EVALUATION.csv
Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv
Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv
